# Classification Evaluation Metrics

When building machine learning classifiers, we need reliable metrics to evaluate how well our models perform. Choosing the right metric is highly dependent on the problem context, dataset balance, and the real-world consequences of our model's errors.

In these notes, we will cover:
1. **Accuracy:** Definition, formula, and its fatal flaw when dealing with imbalanced datasets.
2. **The Confusion Matrix:** Explaining True Positives, True Negatives, False Positives (Type I Error), and False Negatives (Type II Error) for binary and multi-class classification.
3. **Precision, Recall, and the F1 Score:** Why we need them, when to use them, and why the harmonic mean is preferred.
4. **Multi-class Metrics Aggregation:** Macro Average vs. Weighted Average.
5. **Practical Python Demonstration:** Code examples using Scikit-Learn to compute, visualize, and report these metrics.

## 1. Accuracy & The Fatal Flaw of Accuracy

### Definition
**Accuracy** is the simplest and most intuitive classification metric. It measures the ratio of correctly predicted instances to the total number of predictions.

$$\text{Accuracy} = \frac{\text{Number of Correct Predictions}}{\text{Total Number of Predictions}}$$

In terms of binary classification variables:
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

### Is there a "good" accuracy threshold?
There is **no universal threshold** for what constitutes a "good" accuracy score. It is completely problem-dependent:
- A movie recommendation system might consider $85\%$ accuracy excellent.
- A medical diagnostic tool or a self-driving car system might consider $99\%$ accuracy dangerously inadequate, as a $1\%$ failure rate can lead to loss of life.

### The Fatal Flaw: Imbalanced Datasets
Accuracy provides a single overall score, but it completely fails to explain the **nature of errors** and is highly misleading when dealing with **imbalanced datasets** (where one class heavily outweighs the other).

#### Real-World Example: Airport Security
Suppose we are designing a classification system to identify terrorists at an airport security checkpoint.
- **Dataset Profile:** $99.99\%$ of passengers are normal travelers ($10,000$ normal, $1$ threat).
- **The Dummy Model:** A simple model that always predicts "Normal Traveler" (Class 0) for every single passenger.
- **Evaluation:**
  - **Accuracy:** The model gets $10,000$ predictions correct out of $10,001$. This yields a staggering **$99.99\%$ accuracy**.
  - **The Reality:** The system has a **$100\%$ failure rate** at identifying actual threats, making it completely useless and dangerous.

This highlights the critical need for more sophisticated evaluation metrics.

## 2. The Confusion Matrix & Types of Errors

The **Confusion Matrix** is a tabular layout that breaks down the performance of a classifier, allowing us to see exactly where the model is succeeding and where it is making specific errors.

### The $2 \times 2$ Confusion Matrix structure (Binary Classification)

| | Predicted Negative (0) | Predicted Positive (1) |
| :--- | :---: | :---: |
| **Actual Negative (0)** | **True Negative (TN)** | **False Positive (FP)** (Type I Error) |
| **Actual Positive (1)** | **False Negative (FN)** (Type II Error) | **True Positive (TP)** |

- **True Positive (TP):** We predicted positive (1) and it is actually positive.
- **True Negative (TN):** We predicted negative (0) and it is actually negative.
- **False Positive (FP) / Type I Error:** We predicted positive (1) but it is actually negative (e.g., spam filter flagging a normal email as spam).
- **False Negative (FN) / Type II Error:** We predicted negative (0) but it is actually positive (e.g., medical test failing to detect cancer).

### Multi-Class Confusion Matrix
For a classification problem with $N$ classes, the confusion matrix generalizes to an $N \times N$ grid:
- **Diagonal Elements:** Represent correct classification hits for each class.
- **Off-Diagonal Elements:** Represent misclassifications (e.g., row $i$, column $j$ shows how many times class $i$ was incorrectly predicted as class $j$).

## 3. Precision, Recall & F1-Score

To move beyond the limitations of accuracy, we introduce three core metrics that focus on specific aspects of classification quality.

### 1. Precision
**Precision** (also called Positive Predictive Value) answers the question: *Of all instances the model predicted as positive, what proportion was actually positive?*

$$\text{Precision} = \frac{TP}{TP + FP}$$

- **When to prioritize Precision:** When the cost of a **False Positive (Type I Error)** is extremely high.
- **Example:** A spam classifier. If a legitimate email is incorrectly flagged as spam (False Positive), the user might miss a critical business message. We want high precision so that when we say "Spam", we are highly certain.

### 2. Recall
**Recall** (also called Sensitivity or True Positive Rate) answers the question: *Of all actual positive instances, what proportion did the model correctly identify?*

$$\text{Recall} = \frac{TP}{TP + FN}$$

- **When to prioritize Recall:** When the cost of a **False Negative (Type II Error)** is extremely high.
- **Example:** A cancer diagnostic model. If a patient with cancer is classified as healthy (False Negative), they will miss critical life-saving treatment. We want high recall to catch every single potential positive case.

### 3. F1-Score
In many scenarios, we want a balance between Precision and Recall. The **F1-Score** is the **harmonic mean** of Precision and Recall, providing a single metric that represents both.

$$\text{F1-Score} = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

#### Why Harmonic Mean instead of Arithmetic Mean?
The arithmetic mean can be deceptive. If a model has a Precision of $1.0$ and a Recall of $0.0$, the arithmetic mean is $0.5$. However, this model is completely useless (it finds 0 actual positives). 
The harmonic mean penalizes extreme values:
- For Precision = $1.0$ and Recall = $0.0$, the F1-Score is **$0.0$** (properly reflecting a failed model).
- The F1-Score only reaches high values if **both** Precision and Recall are high.

## 4. Multi-Class Aggregations: Macro vs. Weighted Average

When evaluating multi-class classification, metrics are first calculated for each class independently. To combine them into a single score, we use different aggregation strategies:

1. **Macro Average:**
   - Calculates the metric (e.g., Precision) for each class individually, then takes the simple unweighted average.
   - **Formula:** $\text{Macro } P = \frac{P_{\text{class 1}} + P_{\text{class 2}} + \dots + P_{\text{class N}}}{N}$
   - **Characteristics:** Treats all classes equally, regardless of how many samples belong to each class. This is useful when you care equally about minority classes.
2. **Weighted Average:**
   - Calculates the metric for each class individually, then multiplies each by its **Support** (the number of true samples in that class) before averaging.
   - **Formula:** $\text{Weighted } P = \frac{(P_{\text{class 1}} \cdot n_1) + (P_{\text{class 2}} \cdot n_2) + \dots + (P_{\text{class N}} \cdot n_N)}{N_{\text{total}}}$
   - **Characteristics:** Favors performance on classes with larger support. Highly recommended for imbalanced datasets where minority classes shouldn't skew the overall evaluation representation.

## 5. Python Implementation & Demonstration

Let's write Python code to demonstrate these metrics in practice using Scikit-Learn. We will generate an imbalanced dataset representing credit card fraud detection.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    precision_score, 
    recall_score, 
    f1_score, 
    classification_report
)

# Set visual configurations
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)

# Generate an imbalanced dataset: 1000 samples, 90% class 0 (Legitimate), 10% class 1 (Fraudulent)
X, y = make_classification(
    n_samples=1000,
    n_features=5,
    n_informative=3,
    n_classes=2,
    weights=[0.9, 0.1],  # Imbalanced ratio
    random_state=42
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train a simple Logistic Regression model
model = LogisticRegression()
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)

print("Model trained and predictions generated!")
print(f"Test Set Class Distribution - Legitimate (Class 0): {np.sum(y_test == 0)}, Fraudulent (Class 1): {np.sum(y_test == 1)}")

### Plotting and Explaining the Confusion Matrix

Let's compute the confusion matrix and visualize it as a heatmap.

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Extract components
tn, fp, fn, tp = cm.ravel()

# Plot heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Purples', 
    xticklabels=['Predicted Legitimate (0)', 'Predicted Fraud (1)'],
    yticklabels=['Actual Legitimate (0)', 'Actual Fraud (1)']
)
plt.title('Confusion Matrix: Fraud Detection Classifier', fontsize=14, fontweight='bold', pad=15)
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

print("="*50)
print(f"True Negatives (TN):  {tn} (Correctly predicted normal transactions)")
print(f"False Positives (FP): {fp} (Type I Error: Legitimate flagged as Fraud)")
print(f"False Negatives (FN): {fn} (Type II Error: Fraud transaction missed!)")
print(f"True Positives (TP):  {tp} (Correctly predicted fraud)")
print("="*50)

### Calculating Metrics: Manual vs. Scikit-Learn
To verify our mathematical understanding, let's compute Accuracy, Precision, Recall, and F1-Score manually using the variables `tn`, `fp`, `fn`, `tp` and compare them with the outputs of Scikit-Learn.

In [ ]:
# Manual Calculations
manual_accuracy = (tp + tn) / (tp + tn + fp + fn)
manual_precision = tp / (tp + fp) if (tp + fp) > 0 else 0
manual_recall = tp / (tp + fn) if (tp + fn) > 0 else 0
manual_f1 = 2 * (manual_precision * manual_recall) / (manual_precision + manual_recall) if (manual_precision + manual_recall) > 0 else 0

# Sklearn Calculations
sk_accuracy = accuracy_score(y_test, y_pred)
sk_precision = precision_score(y_test, y_pred)
sk_recall = recall_score(y_test, y_pred)
sk_f1 = f1_score(y_test, y_pred)

# Print results
print("="*65)
print(f"{'Metric':<15} | {'Manual Calculation':<20} | {'Scikit-Learn Output':<20}")
print("="*65)
print(f"{'Accuracy':<15} | {manual_accuracy:<20.6f} | {sk_accuracy:<20.6f}")
print(f"{'Precision':<15} | {manual_precision:<20.6f} | {sk_precision:<20.6f}")
print(f"{'Recall':<15} | {manual_recall:<20.6f} | {sk_recall:<20.6f}")
print(f"{'F1-Score':<15} | {manual_f1:<20.6f} | {sk_f1:<20.6f}")
print("="*65)

### The Classification Report

For a quick and comprehensive view of classification performance, Scikit-Learn provides `classification_report`. This is highly recommended as it displays Precision, Recall, and F1-Score for each class, as well as Macro and Weighted averages.

In [ ]:
# Print classification report
report = classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Fraudulent (1)'])
print("Classification Report:")
print(report)

## Summary Checklist for Classification Metrics

1. **Beware Imbalanced Classes:** Never rely solely on **Accuracy** if your target classes are heavily imbalanced.
2. **Prioritize Precision (Type I Error prevention):** Use when flagging a negative instance as positive has high costs (e.g., spam filters).
3. **Prioritize Recall (Type II Error prevention):** Use when missing a positive instance has fatal/high costs (e.g., cancer diagnostics).
4. **Use F1-Score for General Performance:** The harmonic mean represents overall model quality and is sensitive to poor performance in either Precision or Recall.
5. **Use `classification_report`:** Always generate a classification report to view macro and weighted averages and inspect minority class performance.